# SRLG v4 Candidate Derivation (FREEZE-v4 shadow build)

**Project:** Provable Resilience Intelligence (Lakelink Fiber)  
**Tasks:** BUG-DP-069, BUG-QA-074 option b, split out by BUG-ML-V4-GUARD-SRLG-PIPELINE  
**Writes (shadow only):** `derived_srlg_shared_conduit_v4_candidate`, `derived_srlg_common_site_v4_candidate`, `derived_srlg_geo_proximity_v4_candidate`, `derived_srlg_shared_route_v4_candidate`, `derived_shared_risk_group_v4_candidate`, `unified_shared_risk_group_v4_candidate` (view)  
**Reads:** `physical_resource_span_v4_candidate`, `physical_resource_conduit_v4_candidate`, `geographic_site`, `shared_risk_group`

This notebook never writes v3 FROZEN objects. The v3 pipeline stays in `srlg_colocation_derivation`.

* SPAN-* spans keep the v3 rules.
* BBS-* backbone segments join site and proximity SRLGs only at true terminals (seq=1 gives site_a, seq=segment_count gives site_z).
* shared_route groups parallel CDT-BBR conduits that have identical geom_wkt (31 routes).

**Write gate:** writes run only when the `write_enabled` widget is `true`. The default is `false`, which runs the read-only gates only.

Known open issue: BUG-QA-076. The geo_proximity bbox prefilter is not cos(lat)-aware. The logic here mirrors the data built in session 38/39 so that code and data stay consistent until that fix lands.

In [0]:
dbutils.widgets.dropdown("write_enabled", "false", ["false", "true"], "Write v4 candidate tables")
WRITE_ENABLED = dbutils.widgets.get("write_enabled") == "true"
S = "cdm_tmforum.provable_resilience_intelligence"
SP = f"{S}.physical_resource_span_v4_candidate"
CD = f"{S}.physical_resource_conduit_v4_candidate"
GS = f"{S}.geographic_site"

def run_write(sql_text: str):
    """Execute a shadow write only when WRITE_ENABLED. Refuses any non-v4 target."""
    head = sql_text.strip().split("\n", 1)[0]
    assert "_v4_candidate" in head, f"Refusing non-v4 write: {head}"
    if WRITE_ENABLED:
        spark.sql(sql_text)
        print("WROTE:", head[:120])
    else:
        print("SKIPPED (write_enabled=false):", head[:120])

print("WRITE_ENABLED =", WRITE_ENABLED)

In [0]:
spark.sql(f"""CREATE OR REPLACE TEMP VIEW span_site_term AS
SELECT span_id, site_a_id AS site_id FROM {SP} WHERE is_active AND span_id NOT LIKE 'BBS-%'
UNION SELECT span_id, site_z_id FROM {SP} WHERE is_active AND span_id NOT LIKE 'BBS-%'
UNION SELECT span_id, site_a_id FROM {SP} WHERE is_active AND span_id LIKE 'BBS-%' AND segment_seq=1
UNION SELECT span_id, site_z_id FROM {SP} WHERE is_active AND span_id LIKE 'BBS-%' AND segment_seq=segment_count""")

run_write(f"""CREATE OR REPLACE TABLE {S}.derived_srlg_shared_conduit_v4_candidate AS
WITH m AS (
  SELECT sp.span_id, sp.conduit_id,
    CASE WHEN sp.span_id LIKE 'BBS-%' THEN st_y(st_centroid(st_geomfromwkt(sp.geom_wkt))) ELSE (a.latitude+z.latitude)/2 END lat,
    CASE WHEN sp.span_id LIKE 'BBS-%' THEN st_x(st_centroid(st_geomfromwkt(sp.geom_wkt))) ELSE (a.longitude+z.longitude)/2 END lon
  FROM {SP} sp JOIN {GS} a ON sp.site_a_id=a.site_id JOIN {GS} z ON sp.site_z_id=z.site_id
  WHERE sp.is_active AND sp.conduit_id IS NOT NULL),
g AS (SELECT conduit_id, count(*) member_count, avg(lat) rlat, avg(lon) rlon FROM m GROUP BY conduit_id HAVING count(*)>=2)
SELECT concat('SRLG-COND-',m.conduit_id) srlg_id, concat('Shared Conduit ',m.conduit_id) srlg_name, 'shared_conduit' srlg_type,
  m.span_id, m.conduit_id, CAST(NULL AS STRING) site_id, g.rlat risk_latitude, g.rlon risk_longitude, 'critical' severity,
  g.member_count, current_timestamp() derived_at, 'DERIVED' origin, 'v4_candidate' derivation_version
FROM m JOIN g USING (conduit_id)""")

run_write(f"""CREATE OR REPLACE TABLE {S}.derived_srlg_common_site_v4_candidate AS
WITH g AS (SELECT site_id, count(DISTINCT span_id) member_count FROM span_site_term WHERE site_id IS NOT NULL GROUP BY site_id HAVING count(DISTINCT span_id)>=2)
SELECT concat('SRLG-SITE-',t.site_id) srlg_id, concat('Site Colocation ',gs.site_name) srlg_name, 'central_office' srlg_type,
  t.span_id, CAST(NULL AS STRING) conduit_id, t.site_id, gs.latitude risk_latitude, gs.longitude risk_longitude, 'high' severity,
  CAST(g.member_count AS BIGINT) member_count, current_timestamp() derived_at, 'DERIVED' origin, 'v4_candidate' derivation_version
FROM span_site_term t JOIN g USING (site_id) JOIN {GS} gs ON t.site_id=gs.site_id""")

run_write(f"""CREATE OR REPLACE TABLE {S}.derived_srlg_geo_proximity_v4_candidate AS
WITH sp AS (
  SELECT s1.site_id s1, s2.site_id s2,
    6371000*2*ASIN(SQRT(POWER(SIN(RADIANS(s2.latitude-s1.latitude)/2),2)+COS(RADIANS(s1.latitude))*COS(RADIANS(s2.latitude))*POWER(SIN(RADIANS(s2.longitude-s1.longitude)/2),2))) d,
    (s1.latitude+s2.latitude)/2 mlat, (s1.longitude+s2.longitude)/2 mlon
  FROM {GS} s1 JOIN {GS} s2 ON s1.site_id<s2.site_id AND ABS(s1.latitude-s2.latitude)<0.005 AND ABS(s1.longitude-s2.longitude)<0.005
  WHERE s1.is_active AND s2.is_active),
cp AS (SELECT * FROM sp WHERE d<=500),
mem AS (SELECT DISTINCT cp.*, t.span_id FROM cp JOIN span_site_term t ON t.site_id IN (cp.s1, cp.s2)),
g AS (SELECT s1,s2,count(DISTINCT span_id) n FROM mem GROUP BY s1,s2 HAVING count(DISTINCT span_id)>=2)
SELECT concat('SRLG-GEO-',m.s1,'-',m.s2) srlg_id, concat('Proximity ',m.s1,' / ',m.s2,' (',round(m.d,1),'m)') srlg_name, 'geographic_proximity' srlg_type,
  m.span_id, CAST(NULL AS STRING) conduit_id, m.s1 site_id, m.mlat risk_latitude, m.mlon risk_longitude, 'medium' severity,
  CAST(g.n AS BIGINT) member_count, current_timestamp() derived_at, 'DERIVED' origin, 'v4_candidate' derivation_version
FROM mem m JOIN g USING (s1,s2)""")

run_write(f"""CREATE OR REPLACE TABLE {S}.derived_srlg_shared_route_v4_candidate AS
WITH c AS (SELECT conduit_id, geom_wkt, least(site_a_id,site_z_id) sx, greatest(site_a_id,site_z_id) sy FROM {CD} WHERE is_active AND conduit_id LIKE 'CDT-BBR-%'),
grp AS (SELECT geom_wkt, min(sx) sx, min(sy) sy, count(*) n_conduits FROM c GROUP BY geom_wkt HAVING count(*)>=2),
m AS (SELECT grp.sx, grp.sy, grp.n_conduits, sp.span_id, sp.conduit_id,
        st_y(st_centroid(st_geomfromwkt(grp.geom_wkt))) rlat, st_x(st_centroid(st_geomfromwkt(grp.geom_wkt))) rlon
      FROM grp JOIN c ON c.geom_wkt=grp.geom_wkt JOIN {SP} sp ON sp.conduit_id=c.conduit_id AND sp.is_active)
SELECT concat('SRLG-ROUTE-',sx,'-',sy) srlg_id, concat('Shared Backbone Route ',sx,' - ',sy,' (',n_conduits,' parallel conduits)') srlg_name,
  'shared_route' srlg_type, span_id, conduit_id, CAST(NULL AS STRING) site_id, rlat risk_latitude, rlon risk_longitude, 'critical' severity,
  count(*) OVER (PARTITION BY sx,sy) member_count, current_timestamp() derived_at, 'DERIVED' origin, 'v4_candidate' derivation_version
FROM m""")

cols = "srlg_id, srlg_name, srlg_type, span_id, conduit_id, site_id, risk_latitude, risk_longitude, severity, member_count, derived_at, origin, derivation_version"
run_write(f"""CREATE OR REPLACE TABLE {S}.derived_shared_risk_group_v4_candidate AS
SELECT {cols} FROM {S}.derived_srlg_shared_conduit_v4_candidate UNION ALL
SELECT {cols} FROM {S}.derived_srlg_common_site_v4_candidate UNION ALL
SELECT {cols} FROM {S}.derived_srlg_geo_proximity_v4_candidate UNION ALL
SELECT {cols} FROM {S}.derived_srlg_shared_route_v4_candidate""")
# CREATE OR REPLACE drops column comments. Re-apply them from the comment snapshot after any re-run (DEC-PRI-FREEZE-COMMENTS).

In [0]:
run_write(f"""CREATE OR REPLACE VIEW {S}.unified_shared_risk_group_v4_candidate (
  srlg_id COMMENT 'Unique SRLG identifier',
  srlg_name COMMENT 'Descriptive name of the shared risk group',
  srlg_type COMMENT 'Risk type: bridge | rail_crossing | shared_conduit | central_office | tunnel | power_feed | geographic_proximity | shared_route',
  span_id COMMENT 'FK to physical_resource_span_v4_candidate.span_id (member span; SPAN-* or BBS-*)',
  conduit_id COMMENT 'FK to physical_resource_conduit_v4_candidate.conduit_id if conduit/route-based',
  site_id COMMENT 'FK to geographic_site.site_id if site-based (CO/DC risk)',
  risk_latitude COMMENT 'Latitude (WGS84) of the shared-risk point',
  risk_longitude COMMENT 'Longitude (WGS84) of the shared-risk point',
  severity COMMENT 'Risk severity: critical | high | medium | low',
  source COMMENT 'SRLG origin: seed | derived_shared_conduit | derived_common_site | derived_geo_proximity | derived_shared_route')
COMMENT 'Lakelink Fiber unified SRLG memberships, FREEZE-v4 CANDIDATE (BUG-DP-069 + BUG-QA-074). Seed shared_risk_group (unchanged) + v4 derived SRLGs computed on physical_resource_span_v4_candidate, including BBS-* backbone segments and a shared_route SRLG for parallel CDT-BBR conduits on identical geometry. One row per (srlg_id, span_id, source). Shadow build: do not consume until cutover.'
AS SELECT srlg_id,srlg_name,srlg_type,span_id,conduit_id,site_id,risk_latitude,risk_longitude,severity,'seed' FROM {S}.shared_risk_group
UNION ALL SELECT srlg_id,srlg_name,srlg_type,span_id,conduit_id,site_id,risk_latitude,risk_longitude,severity,
  concat('derived_', CASE srlg_type WHEN 'central_office' THEN 'common_site' WHEN 'geographic_proximity' THEN 'geo_proximity' ELSE srlg_type END)
FROM {S}.derived_shared_risk_group_v4_candidate""")

In [0]:
g = spark.sql(f"""
WITH d AS (SELECT * FROM {S}.derived_shared_risk_group_v4_candidate),
multi AS (SELECT conduit_id FROM {SP} WHERE is_active AND conduit_id IS NOT NULL GROUP BY conduit_id HAVING count(*)>=2)
SELECT
  (SELECT count(*) FROM d) AS derived_rows,
  (SELECT count(*) FROM {S}.unified_shared_risk_group_v4_candidate) AS unified_rows,
  (SELECT count(DISTINCT srlg_id) FROM {S}.unified_shared_risk_group_v4_candidate) AS unified_srlgs,
  (SELECT count(*) FROM d LEFT ANTI JOIN {SP} s USING (span_id)) AS g1_span_fk_orphans,
  (SELECT count(*) FROM d WHERE conduit_id IS NOT NULL AND conduit_id NOT IN (SELECT conduit_id FROM {CD})) AS g2_conduit_fk_orphans,
  (SELECT count(*) FROM multi WHERE conduit_id NOT IN (SELECT conduit_id FROM d WHERE srlg_type='shared_conduit')) AS g3_uncovered_multi_conduits,
  (SELECT count(DISTINCT srlg_id) FROM d WHERE srlg_type='shared_route') AS g4_shared_route_srlgs,
  (SELECT count(*) FROM d WHERE span_id LIKE 'BBS-%' AND srlg_type IN ('central_office','geographic_proximity')
     AND span_id NOT IN (SELECT span_id FROM {SP} WHERE segment_seq=1 OR segment_seq=segment_count)) AS g5_bbs_nonterminal_site_members
""")
display(g)
r = g.collect()[0]
assert r.g1_span_fk_orphans == 0 and r.g2_conduit_fk_orphans == 0 and r.g3_uncovered_multi_conduits == 0 and r.g5_bbs_nonterminal_site_members == 0, r
print("All gates passed")